In [0]:
########################
#Install Dependancies
########################
#Install openpyxl so can export as excel (only required to once for your computer)
# %pip install openpyxl
# %pip install pycountry
# dbutils.library.restartPython()

In [0]:
########################
#Setup for Tranformation Tests
########################

#####################
#Define This notebook name
this_notebook_state = "ftpaSubmitted(a)"

#Setup Test Reporting
from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"
#Capture Test Start datetime
run_start_datetime = datetime.now()
#####################

#Setup Notebook Parameters (Defaulting to Payment Pending and Running all tests)
dbutils.widgets.text("state_under_test", "")

#fields_to_exclude : Should be a comma separated list of fields to exclude 
dbutils.widgets.text("fields_to_exclude", "")

# Read parameters
state_under_test = dbutils.widgets.get("state_under_test")
fields_to_exclude = dbutils.widgets.get("fields_to_exclude")

#Set Default Values if not called from another Notebook
if state_under_test == "" :    
    state_under_test = this_notebook_state

#Get Fields to Exclude
if fields_to_exclude != "":    
    fields_to_exclude = [item.strip() for item in fields_to_exclude.split(",")]
else:
    fields_to_exclude = []
    
#List of Fields to Exclude in Child Notebooks Called
child_fields_to_exclude = [""]

#Combine any Fields to Exclude
fields_to_exclude.extend(child_fields_to_exclude)

print(f"Fields to exclude = {str(fields_to_exclude)}")
print(f"Testing State = {state_under_test}")

#Restart Python When needed (when changed tests)
# dbutils.library.restartPython()

#Import Tests
import Test_Functions.FTPA_Submitted_A_Tests as ftpa_a_tests

#Import models.test_result as test_result
from models.test_result import TestResult
# TestResult.test_from_state = "appealSubmitted"

#Import asdict to convert Dataclass to Dictionary
from dataclasses import asdict
import os

#Setup Global Variables
all_test_results = []
current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = current_path.rsplit("/", 1)[0] + "/"
#Below will be replaced eventually to store the reults in a spark table
test_results_path= "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)


In [0]:
#Load Config and Setup Enviorment Variables
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
# print(f"env_code: {lz_key}")  # This won't be redacted
# print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
# print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
  
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
  
# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
audit_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/AUDIT/{state_under_test}"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"
 
 
# Print all variables
# variables = {
#     # "read_hive": read_hive,
    
#     "bronze_path": bronze_path,
#     "silver_path": silver_path,
#     "audit_path": audit_path,
#     "gold_path": gold_path,
#     "key_vault": KeyVault_name,
#     "AppealState": state_under_test
 
# }
 
# display(variables)

import json

#Get Latest Json Folder
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_location = json_location.name
dbutils.fs.ls(f"{gold_path}/{latest_json_location}")

#Set Paths
try: 
    json_path = f"{gold_path}/{latest_json_location}/JSON/"
    # json_path = f"{gold_path}/{latest_json_location}/INVALID_JSON/"
    M1_silver = f"{silver_path}/silver_appealcase_detail"
    M1_bronze = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
    M3_bronze = f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj"
    M2_silver = f"{silver_path}/silver_caseapplicant_detail"
    M3_silver = f"{silver_path}/silver_status_detail"
    C = f"{silver_path}/silver_appealcategory_detail"
    bhc = f"{bronze_path}/bronze_hearing_centres"    
except:
    print(f"Error during fetch: {str(e)}")

#Create and Load Dataframes
json_data = spark.read.format("json").load(json_path)
M1_silver = spark.read.format("delta").load(M1_silver)
M1_bronze = spark.read.format("delta").load(M1_bronze)
M3_bronze = spark.read.format("delta").load(M3_bronze)
M2_silver = spark.read.format("delta").load(M2_silver)
M3_silver = spark.read.format("delta").load(M3_silver)
C = spark.read.format("delta").load(C)
bhc = spark.read.format("delta").load(bhc)


# Load data that isn't loading
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("appealReferenceNumber", StringType(), True),
    StructField("witness1InterpreterSignLanguage", StringType(), True),
    StructField("witness2InterpreterSignLanguage", StringType(), True),
    StructField("witness3InterpreterSignLanguage", StringType(), True),
    StructField("witness4InterpreterSignLanguage", StringType(), True),
    StructField("witness5InterpreterSignLanguage", StringType(), True),
    StructField("witness6InterpreterSignLanguage", StringType(), True),
    StructField("witness7InterpreterSignLanguage", StringType(), True),
    StructField("witness8InterpreterSignLanguage", StringType(), True),
    StructField("witness9InterpreterSignLanguage", StringType(), True),
    StructField("witness10InterpreterSignLanguage", StringType(), True),
    StructField("witness1InterpreterSpokenLanguage", StringType(), True),
    StructField("witness2InterpreterSpokenLanguage", StringType(), True),
    StructField("witness3InterpreterSpokenLanguage", StringType(), True),
    StructField("witness4InterpreterSpokenLanguage", StringType(), True),
    StructField("witness5InterpreterSpokenLanguage", StringType(), True),
    StructField("witness6InterpreterSpokenLanguage", StringType(), True),
    StructField("witness7InterpreterSpokenLanguage", StringType(), True),
    StructField("witness8InterpreterSpokenLanguage", StringType(), True),
    StructField("witness9InterpreterSpokenLanguage", StringType(), True),
    StructField("witness10InterpreterSpokenLanguage", StringType(), True)
])

print (f"Before - Number of Fields : {str(len(json_data.columns))}") 
missing_fields_json = spark.read.schema(schema).json(json_path)
json_data = json_data.join(
    missing_fields_json,
    json_data["appealReferenceNumber"] == missing_fields_json["appealReferenceNumber"],
    "left"
).drop(missing_fields_json["appealReferenceNumber"])
print (f"After - Addding Missing Fields : Count : {str(len(json_data.columns))}") 

In [0]:
# try:
#     #Call Parent Tests - with field exclusion comma separated list
#     if len(child_fields_to_exclude) !=  0:
#         str_child_fields_to_exclude = ",".join(child_fields_to_exclude)
#     else:
#         str_child_fields_to_exclude = ""
    
#     child_notebook_to_run = "Decided_A_TESTS"
#     pp_results = dbutils.notebook.run(child_notebook_to_run, timeout_seconds=10800, arguments={
#         "fields_to_exclude": str_child_fields_to_exclude,
#         "state_under_test": state_under_test
#     })

#     from dataclasses import dataclass
#     @dataclass
#     class TestResult:
#         test_field: str =""
#         status: str =""
#         message: str = ""
#         test_from_state: str=""
#         test_name: str=""    

#     #Convert string of results back into list of TestResults class objects
#     all_test_results = [TestResult(**d) for d in json.loads(pp_results)]
#     print (all_test_results)

# except Exception as e:
#     message =  f"Notebook Execution Failed for: {child_notebook_to_run} -- execution failed with Message : {str(e)}\nFurther Execution Stopped"    
#     dbutils.notebook.exit(message)    

In [0]:
#Used to Quickly reload the function file containing the test, rather than restart python
import importlib
importlib.reload(ftpa_a_tests)

# ftpa Tests

In [0]:
test_data_setup = None
test_df, test_data_setup =  ftpa_a_tests.test_ftpa_init(json_data, M1_bronze, M3_bronze)
if test_data_setup != True:
     all_test_results.append(test_data_setup)

if test_df != None:
     if "ftpaList" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaList_appellant(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaList_respondent(test_df))
     if "ftpaAppellantApplicationDate" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantApplicationDate_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantApplicationDate_test2(test_df))
     if "ftpaAppellantSubmissionOutOfTime" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmissionOutOfTime_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmissionOutOfTime_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmissionOutOfTime_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmissionOutOfTime_test4(test_df))
     if "ftpaAppellantOutOfTimeExplanation" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeExplanation_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeExplanation_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeExplanation_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeExplanation_test4(test_df))
     if "ftpaRespondentApplicationDate" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentApplicationDate_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentApplicationDate_test2(test_df))
     if "ftpaRespondentSubmissionOutOfTime" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmissionOutOfTime_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmissionOutOfTime_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmissionOutOfTime_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmissionOutOfTime_test4(test_df))
     if "ftpaRespondentOutOfTimeExplanation" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeExplanation_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeExplanation_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeExplanation_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeExplanation_test4(test_df))
# display(all_test_results)

# general Tests

In [0]:
import importlib
importlib.reload(ftpa_a_tests)

test_data_setup = None
test_df, test_data_setup =  ftpa_a_tests.test_ftpa_init(json_data, M1_bronze, M3_bronze)
if test_data_setup != True:
     all_test_results.append(test_data_setup)

if test_df != None:
     if "ftpaAppellantSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantSubmitted_test2(test_df))
     if "isFtpaAppellantDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantDocsVisibleInDecided_test2(test_df))
     if "isFtpaAppellantDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaAppellantOotDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInDecided_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInDecided_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInDecided_test4(test_df))
     if "isFtpaAppellantOotDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInSubmitted_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInSubmitted_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotDocsVisibleInSubmitted_test4(test_df))
     if "isFtpaAppellantGroundsDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantGroundsDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantGroundsDocsVisibleInDecided_test2(test_df))
     if "isFtpaAppellantEvidenceDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantEvidenceDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantEvidenceDocsVisibleInDecided_test2(test_df))
     if "isFtpaAppellantGroundsDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantGroundsDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantGroundsDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaAppellantEvidenceDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantEvidenceDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantEvidenceDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaAppellantOotExplanationVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInDecided_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInDecided_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInDecided_test4(test_df))
     if "isFtpaAppellantOotExplanationVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInSubmitted_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInSubmitted_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaAppellantOotExplanationVisibleInSubmitted_test4(test_df))
     if "ftpaRespondentSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentSubmitted_test2(test_df))
     if "isFtpaRespondentDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentDocsVisibleInDecided_test2(test_df))
     if "isFtpaRespondentDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaRespondentOotDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInDecided_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInDecided_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInDecided_test4(test_df))
     if "isFtpaRespondentOotDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInSubmitted_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInSubmitted_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotDocsVisibleInSubmitted_test4(test_df))
     if "isFtpaRespondentGroundsDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentGroundsDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentGroundsDocsVisibleInDecided_test2(test_df))
     if "isFtpaRespondentEvidenceDocsVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentEvidenceDocsVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentEvidenceDocsVisibleInDecided_test2(test_df))
     if "isFtpaRespondentGroundsDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentGroundsDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentGroundsDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaRespondentEvidenceDocsVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentEvidenceDocsVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentEvidenceDocsVisibleInSubmitted_test2(test_df))
     if "isFtpaRespondentOotExplanationVisibleInDecided" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInDecided_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInDecided_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInDecided_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInDecided_test4(test_df))
     if "isFtpaRespondentOotExplanationVisibleInSubmitted" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInSubmitted_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInSubmitted_test2(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInSubmitted_test3(test_df))
          all_test_results.append(ftpa_a_tests.test_isFtpaRespondentOotExplanationVisibleInSubmitted_test4(test_df))
# display(all_test_results)

# document Tests

In [0]:
import importlib
importlib.reload(ftpa_a_tests)

test_data_setup = None
test_df, test_data_setup =  ftpa_a_tests.test_ftpa_init(json_data, M1_bronze, M3_bronze)
if test_data_setup != True:
     all_test_results.append(test_data_setup)

if test_df != None:
     if "ftpaAppellantDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantDocuments_test2(test_df))
     if "ftpaRespondentDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentDocuments_test2(test_df))
     if "ftpaAppellantGroundsDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantGroundsDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantGroundsDocuments_test2(test_df))
     if "ftpaRespondentGroundsDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentGroundsDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentGroundsDocuments_test2(test_df))
     if "ftpaAppellantEvidenceDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantEvidenceDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantEvidenceDocuments_test2(test_df))
     if "ftpaRespondentEvidenceDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentEvidenceDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentEvidenceDocuments_test2(test_df))
     if "ftpaAppellantOutOfTimeDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaAppellantOutOfTimeDocuments_test2(test_df))
     if "ftpaRespondentOutOfTimeDocuments" not in fields_to_exclude:
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeDocuments_test1(test_df))
          all_test_results.append(ftpa_a_tests.test_ftpaRespondentOutOfTimeDocuments_test2(test_df))
# display(all_test_results)

# Default Mappings

In [0]:
test_data_setup = None
test_df, test_data_setup =  ftpa_a_tests.test_default_mapping_init(json_data)
if test_data_setup != True:
     all_test_results.append(test_data_setup)

if test_df != None:
    all_test_results.extend(ftpa_a_tests.test_ftpa_a_defaultValues(test_df, fields_to_exclude))
# display(all_test_results)

In [0]:
#######################
#PROCESS RESULTS
#######################
from collections import defaultdict
from datetime import datetime

#Convert Results Classes to string
all_test_results_string = json.dumps([asdict(r) for r in all_test_results])
lines = []

#Print Results
grouped = defaultdict(list)
for r in all_test_results:
        grouped[r.test_from_state].append(r)
total_tests = len(all_test_results)
total_passed = sum(1 for r in all_test_results if r.status == "PASS")
total_failed = sum(1 for r in all_test_results if r.status == "FAIL")

lines.append("TEST EXECUTION REPORT")
lines.append(f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
lines.append("-" * 80)
lines.append(f"Total tests : {total_tests}")
lines.append(f"Passed      : {total_passed}")
lines.append(f"Failed      : {total_failed}")
lines.append("-" * 80)

for state in sorted(grouped):  
    state_results = grouped[state]
    state_passed = sum(1 for r in state_results if r.status == "PASS")
    state_failed = sum(1 for r in state_results if r.status == "FAIL")
    lines.append(f"\nSTATE: {state}")
    lines.append(f"Tests: {len(state_results)} | Passed: {state_passed} | Failed: {state_failed}")
    lines.append("-" * 60)

print("\n".join(lines))

#Check If if this notebook is parent notebook(so produce report), or has been called by another(return data to parent)
if state_under_test == this_notebook_state:
    #Call Print Test Report Notebook
    dbutils.notebook.run( base_path + "Shared_Notebooks/Common_Produce_TestResults", 600, {
    "all_test_results_string":all_test_results_string,
    "state_under_test":state_under_test,
    "base_path":base_path,
    "test_results_path": test_results_path,
    "run_user":run_user,
    "run_tag":run_tag,
    "run_by_automation_name": run_by_automation_name,
    "run_start_datetime": run_start_datetime.strftime("%Y-%m-%d %H:%M:%S"), 
    })
    
#else return to parent notebook
else:
    print(f"Exiting Notebook : {this_notebook_state} and returning Data to Parent notebook : {state_under_test}")
    dbutils.notebook.exit(all_test_results_string)